# 04 - Kampanye tuning hyperparameter

Menjalankan grid yang dirancang di `tuning_grids/`. Prinsipnya human-in-the-loop:
tidak ada pencarian otomatis, urutan konfigurasi ditentukan manusia, dan setiap
baris hasil membawa kolom `catatan` yang merekam alasan konfigurasi itu dicoba.

Seleksi memakai split validation. Split test tidak disentuh sama sekali di
notebook ini.

Metode: grid kombinatorial untuk sumbu yang saling terkait, coordinate descent
untuk sumbu yang independen. Untuk RM-a, `lr`, `epochs`, dan `batch` bersama-sama
menentukan lintasan optimasi (batch 32 pada 5 epoch memberi separuh jumlah
langkah pembaruan dibanding batch 16), sehingga ketiganya harus digrid bersama;
`warmup_ratio` dan `weight_decay` efektif independen sehingga cukup dicoba satu
per satu di sel pemenang.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [1]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
runner.write_hardware()
print("device   :", runner.device)
print("keluaran :", runner.out_dir)

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-14 16:50:56,541 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device   : cuda
keluaran : /workspace/indobert-with-rac/outputs/tuning


## 1. Kalibrasi biaya

Jalankan satu konfigurasi RM-a lebih dulu untuk mengukur waktu dan memori
sesungguhnya di mesin ini, sebelum mempertaruhkan berjam-jam pada grid penuh.
Kalau memori kurang, turunkan `MICRO_BATCH` di `.env`; batch efektif tidak
berubah karena selisihnya ditutup akumulasi gradien.

In [2]:
kalibrasi = runner.run(
    "rma",
    {"lr": 2e-5, "epochs": 5, "batch": 16, "warmup_ratio": 0.1, "weight_decay": 0.01},
    note="kalibrasi biaya: baseline kanonik, sekaligus run #1 grid",
)

per_run = kalibrasi["train_time_s"]
print(f"satu run RM-a: {per_run:.0f} s | peak {kalibrasi['peak_mem_mb']:.0f} MB")
print(f"perkiraan 26 run RM-a: {per_run * 26 / 60:.0f} menit")

2026-09-14 16:50:57,638 | INFO     | src.services.campaign | [rma] RUN #1 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51823.33it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 16:50:58,871 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 16:51:24,359 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9586
2026-09-14 16:51:50,047 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9682
2026-09-14 16:52:16,067 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9762
2026-09-14 16:52:41,696 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9762
2026-09-14 16:53:07,142 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9784
2026-09-14 16:53:07,596 | INFO     | src.services.run_log | Juara baru untuk rma: val F1-macro 0.9784 (sebelumnya -1.0000)
2026-09-14 16:53:09,556 | INFO     | src.services.campaign | [rma] RUN #1 val F1-macro 0.9784 (epoch terbaik 5)
satu run RM-a: 128 s | peak 2334 MB
perkiraan 26 run RM-a: 56 menit


## 2. Muat rancangan grid

In [3]:
GRID_DIR = settings.data_dir.parent / "tuning_grids"

def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]

for berkas in sorted(GRID_DIR.glob("*.csv")):
    print(f"  {berkas.name}: {len(pd.read_csv(berkas))} konfigurasi")

  RMA_TUNING_GRID.csv: 24 konfigurasi
  RMA_TUNING_GRID_STAGE2.csv: 2 konfigurasi
  RMB_TUNING_GRID.csv: 4 konfigurasi
  RMB_TUNING_GRID_STAGE1B.csv: 2 konfigurasi
  RMB_TUNING_GRID_STAGE2.csv: 15 konfigurasi
  RMB_TUNING_GRID_STAGE3.csv: 6 konfigurasi
  RMC_TUNING_GRID.csv: 66 konfigurasi
  RMC_TUNING_GRID_STAGE2.csv: 1 konfigurasi


## Melanjutkan kampanye yang terputus

`run_batch` menyaring konfigurasi yang sudah ada di riwayat secara default
(`resume=True`), sehingga sel batch di bawah aman dijalankan ulang apa adanya
setelah kernel mati, listrik padam, atau proses dihentikan. Yang sudah selesai
dilewati, penomoran run berlanjut, dan `best.json` tetap terjaga.

Yang hilang saat terputus hanyalah run yang sedang berjalan saat itu; run yang
sudah selesai ditulis atomik ke `runs_{skenario}.csv` begitu selesai.

Perbandingan memakai konfigurasi LENGKAP setelah nilai default diisi, dan untuk
RM-a `micro_batch` dinormalkan ke nilai efektifnya (`min(batch, micro_batch)`) —
nilai itulah yang menentukan ukuran batch di `DataLoader`.

Sel di bawah memperlihatkan apa yang tersisa sebelum batch dijalankan.

In [4]:
def sisa(skenario: str, berkas: str) -> None:
    permintaan = muat_grid(berkas)
    tersisa = runner.pending_requests(skenario, permintaan)
    print(f"{berkas:34s} {len(permintaan) - len(tersisa):>3d}/{len(permintaan)} selesai, "
          f"{len(tersisa)} tersisa")

sisa("rma", "RMA_TUNING_GRID.csv")
sisa("rma", "RMA_TUNING_GRID_STAGE2.csv")
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    sisa("rmb", berkas)
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    sisa("rmc", berkas)

RMA_TUNING_GRID.csv                  0/24 selesai, 24 tersisa
RMA_TUNING_GRID_STAGE2.csv           0/2 selesai, 2 tersisa
RMB_TUNING_GRID.csv                  0/4 selesai, 4 tersisa
RMB_TUNING_GRID_STAGE1B.csv          0/2 selesai, 2 tersisa
RMB_TUNING_GRID_STAGE2.csv           0/15 selesai, 15 tersisa
RMB_TUNING_GRID_STAGE3.csv           0/6 selesai, 6 tersisa
RMC_TUNING_GRID.csv                  0/66 selesai, 66 tersisa
RMC_TUNING_GRID_STAGE2.csv           0/1 selesai, 1 tersisa


## 3. RM-a

Grid tahap 1 menggarap tiga sumbu yang saling terkait. Konfigurasi yang gagal
diisolasi ke `runs_rma_errors.csv` dan tidak menghentikan sisa antrean.

In [5]:
hasil_rma = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID.csv"),
                             batch_id="rma_tahap1_grid")
hasil_rma.nlargest(10, "val_f1_macro")[
    ["run_id", "lr", "epochs", "batch", "val_f1_macro", "val_f1_judi",
     "train_time_s", "is_tie_with_best"]
]

2026-09-14 16:53:09,595 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 24 konfigurasi akan dijalankan
2026-09-14 16:53:09,595 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 1/24 (perkiraan sisa 0.0 menit)
2026-09-14 16:53:09,597 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 1 run
2026-09-14 16:53:09,597 | INFO     | src.services.campaign | [rma] RUN #2 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54510.61it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 16:53:10,602 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 16:53:31,868 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9584
2026-09-14 16:53:53,340 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9587
2026-09-14 16:54:15,040 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9747
2026-09-14 16:54:57,920 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9749
2026-09-14 16:54:58,283 | INFO     | src.services.campaign | [rma] RUN #2 val F1-macro 0.9749 (epoch terbaik 5)
2026-09-14 16:54:58,284 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 2/24 (perkiraan sisa 41.7 menit)
2026-09-14 16:54:58,287 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 2 run
2026-09-14 16:54:58,287 | INFO     | src.services.campaign | [rma] RUN #3 {'lr': 2e-05, 'epochs': 3, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22192.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 16:54:59,271 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 16:55:20,574 | INFO     | src.services.training | [RM-a] epoch 1/3 val F1-macro 0.9595
2026-09-14 16:55:41,987 | INFO     | src.services.training | [RM-a] epoch 2/3 val F1-macro 0.9659
2026-09-14 16:56:26,003 | INFO     | src.services.training | [RM-a] epoch 1/8 val F1-macro 0.9543
2026-09-14 16:56:47,614 | INFO     | src.services.training | [RM-a] epoch 2/8 val F1-macro 0.9215
2026-09-14 16:57:08,884 | INFO     | src.services.training | [RM-a] epoch 3/8 val F1-macro 0.9710
2026-09-14 16:57:30,529 | INFO     | src.services.training | [RM-a] epoch 4/8 val F1-macro 0.9683
2026-09-14 16:57:51,852 | INFO     | src.services.training | [RM-a] epoch 5/8 val F1-macro 0.9751
2026-09-14 16:58:13,371 | INFO     | src.services.training | [RM-a] epoch 6/8 val F1-macro 0.9763
2026-09-14 16:58:34,843 | INFO     | src.services.training | [RM-a] epoch 7/8 val F1-macro 0.9751
2026-09-14 16:58:56,

...

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 56149.78it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 17:31:06,042 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 17:31:22,541 | INFO     | src.services.training | [RM-a] epoch 1/3 val F1-macro 0.9713
2026-09-14 17:31:39,119 | INFO     | src.services.training | [RM-a] epoch 2/3 val F1-macro 0.9650
2026-09-14 17:31:55,493 | INFO     | src.services.training | [RM-a] epoch 3/3 val F1-macro 0.9644
2026-09-14 17:31:55,650 | INFO     | src.services.campaign | [rma] RUN #24 val F1-macro 0.9713 (epoch terbaik 1)
2026-09-14 17:31:55,651 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 24/24 (perkiraan sisa 1.7 menit)
2026-09-14 17:31:55,654 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 24 run
2026-09-14 17:31:55,654 | INFO     | src.services.campaign | [rma] RUN #25 {'lr': 5e-05, 'epochs': 8, 'batch': 32, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 56331.68it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 17:31:56,520 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 17:32:12,935 | INFO     | src.services.training | [RM-a] epoch 1/8 val F1-macro 0.9562
2026-09-14 17:32:29,608 | INFO     | src.services.training | [RM-a] epoch 2/8 val F1-macro 0.9617
2026-09-14 17:32:46,153 | INFO     | src.services.training | [RM-a] epoch 3/8 val F1-macro 0.9733
2026-09-14 17:33:02,773 | INFO     | src.services.training | [RM-a] epoch 4/8 val F1-macro 0.9381
2026-09-14 17:33:19,149 | INFO     | src.services.training | [RM-a] epoch 5/8 val F1-macro 0.9726
2026-09-14 17:33:35,613 | INFO     | src.services.training | [RM-a] epoch 6/8 val F1-macro 0.9762
2026-09-14 17:33:52,396 | INFO     | src.services.training | [RM-a] epoch 7/8 val F1-macro 0.9736
2026-09-14 17:34:08,791 | INFO     | src.services.training | [RM-a] epoch 8/8 val F1-macro 0.9760
2026-09-14 17:34:08,965 | INFO     | src.services.campaign | [rma] RUN #25 val F1-macro 0.9762 (epoch terbaik 6)
2026-

,run_id,lr,epochs,batch,val_f1_macro,val_f1_judi,train_time_s,is_tie_with_best
20,22,0.00003,8,32,0.983223,0.972549,132.18,True
12,14,0.00002,5,32,0.982160,0.970874,82.95,False
17,19,0.00001,8,32,0.979597,0.966601,132.17,False
19,21,0.00003,3,32,0.979597,0.966601,49.31,False
3,5,0.00001,5,16,0.978495,0.964844,107.11,True
18,20,0.00003,5,32,0.978430,0.964706,82.79,False
5,7,0.00001,8,16,0.977403,0.963107,170.86,True
1,3,0.00002,3,16,0.977266,0.962818,64.15,True
14,16,0.00002,8,32,0.977126,0.962525,131.59,False
2,4,0.00002,8,16,0.976321,0.961390,171.34,False


Baca grid sebagai permukaan, bukan daftar. Heatmap `lr x epochs` per nilai
`batch` di `outputs/tuning/figures/` memperlihatkan apakah learning rate optimal
ikut bergeser saat batch berubah. Pemenang yang duduk di tepi grid adalah sinyal
untuk melebarkan rentang, bukan untuk langsung dikunci.

Selisih di bawah 0,15 pp dihitung seri karena hanya ada satu seed; pada kondisi
seri, pilih konfigurasi yang lebih murah.

In [6]:
hasil_rma_tahap2 = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID_STAGE2.csv"),
                                    batch_id="rma_tahap2_coordinate")
hasil_rma_tahap2[["run_id", "warmup_ratio", "weight_decay", "val_f1_macro",
                  "delta_vs_best_f1_macro_pp", "is_tie_with_best"]]

2026-09-14 17:34:09,416 | INFO     | src.services.campaign | Batch rma_tahap2_coordinate: 2 konfigurasi akan dijalankan
2026-09-14 17:34:09,417 | INFO     | src.services.campaign | Batch rma_tahap2_coordinate: 1/2 (perkiraan sisa 0.0 menit)
2026-09-14 17:34:09,419 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 25 run
2026-09-14 17:34:09,419 | INFO     | src.services.campaign | [rma] RUN #26 {'lr': 2e-05, 'epochs': 5, 'batch': 32, 'warmup_ratio': 0.0, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 55433.78it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 17:34:10,517 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 17:34:26,942 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9721
2026-09-14 17:34:43,597 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9659
2026-09-14 17:34:59,961 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9694
2026-09-14 17:35:16,357 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9750
2026-09-14 17:35:33,008 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9773
2026-09-14 17:35:33,418 | INFO     | src.services.campaign | [rma] RUN #26 val F1-macro 0.9773 (epoch terbaik 5)
2026-09-14 17:35:33,423 | INFO     | src.services.campaign | Batch rma_tahap2_coordinate: 2/2 (perkiraan sisa 1.4 menit)
2026-09-14 17:35:33,426 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 26 run
2026-09-14 17:35:33,426 | INFO     | src.services.campaign | [rma] RUN #27 {'lr': 

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57381.17it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 17:35:34,421 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 17:35:50,859 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9697
2026-09-14 17:36:07,437 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9675
2026-09-14 17:36:23,875 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9782
2026-09-14 17:36:40,419 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9822
2026-09-14 17:36:57,171 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9783
2026-09-14 17:36:57,328 | INFO     | src.services.campaign | [rma] RUN #27 val F1-macro 0.9822 (epoch terbaik 4)
2026-09-14 17:36:57,923 | INFO     | src.services.campaign | Batch rma_tahap2_coordinate selesai: 2/2 berhasil dalam 2.8 menit


,run_id,warmup_ratio,weight_decay,val_f1_macro,delta_vs_best_f1_macro_pp,is_tie_with_best
0,26,0.0,0.01,0.977266,-0.5957,False
1,27,0.1,0.10,0.982160,-0.1063,True


## 4. RM-b

In [7]:
hasil_rmb = []
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    hasil_rmb.append(runner.run_batch("rmb", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmb, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
     "val_f1_macro", "train_time_s", "trainable_params"]
]

2026-09-14 17:36:57,932 | INFO     | src.services.campaign | Batch rmb_tuning_grid: 4 konfigurasi akan dijalankan
2026-09-14 17:36:57,932 | INFO     | src.services.campaign | Batch rmb_tuning_grid: 1/4 (perkiraan sisa 0.0 menit)
2026-09-14 17:36:57,933 | INFO     | src.services.campaign | [rmb] RUN #1 {'head_arch': 'linear', 'hidden_dim': 256, 'epochs': 5, 'lr': 0.0002, 'dropout': 0.1, 'weight_decay': 0.01, 'batch': 32, 'seed': 42}
2026-09-14 17:36:57,933 | INFO     | src.services.features | Ekstraksi fitur beku dengan encoder indobenchmark/indobert-base-p2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57670.59it/s]

2026-09-14 17:36:58,782 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3


2026-09-14 17:37:04,073 | INFO     | src.services.features |   train: (6588, 768)
2026-09-14 17:37:05,161 | INFO     | src.services.features |   val: (1402, 768)
2026-09-14 17:37:06,255 | INFO     | src.services.features |   test: (1405, 768)
2026-09-14 17:37:06,374 | INFO     | src.services.features | Fitur beku disimpan ke /workspace/indobert-with-rac/outputs/tuning/features/indobenchmark__indobert-base-p2
2026-09-14 17:37:07,140 | INFO     | src.services.run_log | Juara baru untuk rmb: val F1-macro 0.9125 (sebelumnya -1.0000)
2026-09-14 17:37:07,274 | INFO     | src.services.campaign | [rmb] RUN #1 val F1-macro 0.9125 (epoch terbaik 5)
2026-09-14 17:37:07,275 | INFO     | src.services.campaign | Batch rmb_tuning_grid: 2/4 (perkiraan sisa 0.5 menit)
2026-09-14 17:37:07,277 | INFO     | src.services.run_log | Riwayat runs_rmb.csv dimuat: 1 run
2026-09-14 17:37:07,278 | INFO     | src.services.campaign | [rmb] RUN #2 {'head_arch': 'mlp', 'hidden_dim': 128, 'epochs': 5, 'lr': 0.0002, 'd

,run_id,head_arch,hidden_dim,lr,epochs,dropout,val_f1_macro,train_time_s,trainable_params
23,24,mlp,1024,0.0010,10,0.1,0.965301,9.03,789506
18,19,mlp,1024,0.0010,10,0.1,0.964980,9.06,789506
19,20,mlp,1024,0.0010,20,0.1,0.964980,10.58,789506
20,21,mlp,1024,0.0010,30,0.1,0.964980,12.12,789506
15,16,mlp,1024,0.0005,20,0.1,0.964049,10.56,789506
16,17,mlp,1024,0.0005,30,0.1,0.964049,12.09,789506
22,23,mlp,1024,0.0010,10,0.3,0.963828,9.06,789506
12,13,mlp,1024,0.0002,30,0.1,0.963716,12.06,789506
26,27,mlp,1024,0.0010,10,0.1,0.963603,8.49,789506
11,12,mlp,1024,0.0002,20,0.1,0.962908,10.56,789506


## 5. RM-c

RM-c mewarisi head RM-b terbaik, jadi kampanye ini harus dijalankan SETELAH
RM-b selesai. Karena tidak ada training sama sekali, ratusan kombinasi
`alpha x k` selesai dalam hitungan detik.

In [8]:
hasil_rmc = []
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    hasil_rmc.append(runner.run_batch("rmc", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmc, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "alpha", "k", "weighting", "val_f1_macro", "val_f1_judi", "eval_time_s"]
]

2026-09-14 17:38:08,767 | INFO     | src.services.campaign | Batch rmc_tuning_grid: 66 konfigurasi akan dijalankan
2026-09-14 17:38:08,767 | INFO     | src.services.campaign | Batch rmc_tuning_grid: 1/66 (perkiraan sisa 0.0 menit)
2026-09-14 17:38:08,768 | INFO     | src.services.campaign | [rmc] RUN #1 {'alpha': 0.0, 'k': 1, 'weighting': 'similarity'}
2026-09-14 17:38:08,788 | INFO     | src.services.rac | Indeks FAISS dibangun: 6588 vektor berdimensi 768
2026-09-14 17:38:08,906 | INFO     | src.services.run_log | Juara baru untuk rmc: val F1-macro 0.9653 (sebelumnya -1.0000)
2026-09-14 17:38:08,908 | INFO     | src.services.campaign | [rmc] RUN #1 val F1-macro 0.9653
2026-09-14 17:38:08,910 | INFO     | src.services.campaign | Batch rmc_tuning_grid: 2/66 (perkiraan sisa 0.2 menit)
2026-09-14 17:38:08,918 | INFO     | src.services.run_log | Riwayat runs_rmc.csv dimuat: 1 run
2026-09-14 17:38:08,919 | INFO     | src.services.campaign | [rmc] RUN #2 {'alpha': 0.0, 'k': 3, 'weighting': '

,run_id,alpha,k,weighting,val_f1_macro,val_f1_judi,eval_time_s
12,13,0.2,1,similarity,0.969995,0.950884,0.18
14,15,0.2,5,similarity,0.969903,0.950690,0.17
66,67,0.2,5,uniform,0.969903,0.950690,0.05
24,25,0.4,1,similarity,0.968747,0.948819,0.09
13,14,0.2,3,similarity,0.968651,0.948617,0.10
25,26,0.4,3,similarity,0.968554,0.948413,0.18
26,27,0.4,5,similarity,0.968554,0.948413,0.11
27,28,0.4,10,similarity,0.968456,0.948207,0.27
32,33,0.5,5,similarity,0.968358,0.948000,0.07
18,19,0.3,1,similarity,0.967496,0.946746,0.20


## 6. Juara tiap skenario

In [9]:
import json

best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1-macro {entri['val_f1_macro']:.4f}")
    print(f"     {entri['config']}\n")

rma: run #22 | val F1-macro 0.9832
     {'lr': 3e-05, 'epochs': 8, 'batch': 32, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}

rmb: run #24 | val F1-macro 0.9653
     {'head_arch': 'mlp', 'hidden_dim': 1024, 'epochs': 10, 'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'batch': 32, 'seed': 42}

rmc: run #13 | val F1-macro 0.9700
     {'alpha': 0.2, 'k': 1, 'weighting': 'similarity'}



In [10]:
summary = json.loads((OUT_DIR / "tuning_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "generated_at": "2026-09-14 17:38:27",
  "scenarios": {
    "rma": {
      "n_runs": 27,
      "first_run_at": "2026-09-14 16:53:07",
      "last_run_at": "2026-09-14 17:36:57",
      "n_batches": 2,
      "best_run_id": 22,
      "best_val_f1_macro": 0.9832230712686121
    },
    "rmb": {
      "n_runs": 27,
      "first_run_at": "2026-09-14 17:37:06",
      "last_run_at": "2026-09-14 17:38:07",
      "n_batches": 4,
      "best_run_id": 24,
      "best_val_f1_macro": 0.9653006725992406
    },
    "rmc": {
      "n_runs": 67,
      "first_run_at": "2026-09-14 17:38:08",
      "last_run_at": "2026-09-14 17:38:27",
      "n_batches": 2,
      "best_run_id": 13,
      "best_val_f1_macro": 0.9699954201283221
    }
  }
}


## Ringkasan

Seluruh angka di atas berasal dari split validation. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.

Kalau ingin menambah konfigurasi setelah membaca hasil, panggil `runner.run`
atau `runner.run_batch` lagi: riwayat menumpuk dan penomoran run berlanjut.